# Lily 1.5b v0.3 GGUF Conversion & Quantization — Google Colab
**Builds `llama.cpp` binaries, converts Lily-1.5b-v0.3 to GGUF format, and runs K-quantization**

Clones `llama.cpp`, patches system prompt configurations, converts model weights to `F16.gguf`, executes K-quantization (`Q4_K_M`, `Q5_K_M`, `Q8_0`), and uploads GGUF files to `abhinav0231/Lily-1.5b-v0.3-GGUF`.

## Cell 1 — Clone `llama.cpp` & Compile Binaries

In [ ]:
# ==============================================================================
# Cell 1 — Build llama.cpp Binary Tools via CMake
# ==============================================================================
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!mkdir build && cd build && cmake .. && make -j4
%cd ..
print("✅ llama.cpp binaries built successfully")

## Cell 2 — Download Base Model (`abhinav0231/Lily-1.5b-v0.3`)

In [ ]:
# ==============================================================================
# Cell 2 — Download Model Weights from Hugging Face Hub
# ==============================================================================
from huggingface_hub import snapshot_download

repo_id = "abhinav0231/Lily-1.5b-v0.3"
local_dir = "/content/Lily-1.5b-v0.3"

print(f"Downloading model snapshot: {repo_id} ...")
snapshot_download(repo_id=repo_id, local_dir=local_dir)
print(f"✅ Model downloaded to: {local_dir}")

## Cell 3 — Convert PyTorch Model to F16 GGUF

In [ ]:
# ==============================================================================
# Cell 3 — Execute convert_hf_to_gguf.py
# ==============================================================================
!python llama.cpp/convert_hf_to_gguf.py /content/Lily-1.5b-v0.3 --outtype f16 --outfile /content/Lily-1.5b-v0.3-F16.gguf
print("✅ F16 GGUF file generated")

## Cell 4 — Execute K-Quantization (`Q4_K_M`, `Q5_K_M`, `Q8_0`)

In [ ]:
# ==============================================================================
# Cell 4 — Quantize GGUF to 4-bit, 5-bit, and 8-bit Variants
# ==============================================================================
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q4_K_M.gguf Q4_K_M
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q5_K_M.gguf Q5_K_M
!llama.cpp/build/bin/llama-quantize /content/Lily-1.5b-v0.3-F16.gguf /content/Lily-1.5b-v0.3-Q8_0.gguf Q8_0
print("✅ Quantization complete")

## Cell 5 — Upload GGUF Package to Hugging Face Hub

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face)
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"
